# QuantumPulse - end-to-end demonstration

This notebook runs the whole monitoring pipeline on the reference circuit:

1. show the reference program and its five heartbeat checkpoints,
2. verify a **healthy** round (expected: *integrity OK*),
3. inject a controlled **tampering** scenario,
4. **localise** the change by differential testing of single-gate hypotheses,
5. print the resulting **evidence report**.

Everything runs on the Qiskit Aer simulator against the calibrated baseline in
`data/baseline.json` (ideal noise, 4096 shots, `alpha = 0.005` per checkpoint).
To rebuild that baseline, run `PYTHONPATH=src python -m quantumpulse.cli calibrate --runs 200`.

The full measured evaluation (117 scenarios x 3 trials x 4 noise profiles) is in
[`docs/results.md`](../docs/results.md); the method is in
[`docs/method.md`](../docs/method.md).


In [ ]:
import sys
from pathlib import Path

# Run from the repository root or from `notebooks/` - both work.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from quantumpulse.backends import AerBackend
from quantumpulse.calibration import BaselineProfile
from quantumpulse.circuits import reference_program
from quantumpulse.config import NOISE_PROFILES
from quantumpulse.evidence import build_report
from quantumpulse.execution import run_heartbeats
from quantumpulse.localization import localize
from quantumpulse.tampering import apply_tamper, build_tamper_catalog
from quantumpulse.verification import verify_heartbeats

program = reference_program()
backend = AerBackend(noise_config=NOISE_PROFILES["ideal"], seed=7)
baseline = BaselineProfile.load(ROOT / "data" / "baseline.json")

print(f"program : {program.name}  ({program.num_qubits} qubits, {len(program.gates)} gates)")
print(f"baseline: {len(baseline.checkpoints)} checkpoints, {baseline.shots} shots, alpha={baseline.alpha}")


## 1. The reference program and its heartbeat checkpoints

Gates carry stable ids (`g00`...`g15`) so removing a gate never shifts a checkpoint position. Each checkpoint is a prefix execution of the program.

In [ ]:
print("gates")
for gate in program.gates:
    print(f"  {gate.gid}  {gate.op}({','.join(str(q) for q in gate.qubits)})"
          f"{tuple(round(p, 4) for p in gate.params)}  [{gate.stage}]")

print()
print("checkpoints")
for cp in program.checkpoints:
    print(f"  {cp.cp_id:<9} {cp.name:<22} after gate {cp.position}")


## 2. A healthy round

Every checkpoint is compared against the calibrated healthy null with a total-variation distance. The threshold is the `1 - alpha` quantile of that null, computed leave-one-out.

In [ ]:
healthy = run_heartbeats(program, backend, shots=baseline.shots, seed=101)
healthy_verdict = verify_heartbeats(healthy.observations, baseline, program=program)
print("healthy round :", "FLAGGED" if healthy_verdict.flagged else "INTEGRITY OK",
      f"({healthy.total_seconds:.2f} s)")


## 3. Inject a tampering scenario

The catalogue contains 117 controlled single-gate modifications (replacements, removals, angle alterations). The angle of `g11` is altered by `+pi/2`; the deviation is only visible from `CP-OUT` onwards, which is exactly why intermediate checkpoints matter.

In [ ]:
catalog = build_tamper_catalog(program)
tamper = next(t for t in catalog if t.tamper_id == "g11:alter:+1.5708")
tampered = apply_tamper(program, tamper)

heartbeat = run_heartbeats(tampered, backend, shots=baseline.shots, seed=202)
verdict = verify_heartbeats(heartbeat.observations, baseline, program=tampered)

print(f"scenario : {tamper.tamper_id}  ({tamper.description})")
print("tampered round:", "FLAGGED" if verdict.flagged else "not flagged")
for v in verdict.verdicts:
    print(f"  {v.cp_id:<9} D_TV={v.observed_tvd:.4f}  threshold={v.threshold:.4f}"
          f"  p(null)={v.null_p_value:.4f}  noise_floor={v.noise_floor:.4f}"
          f"  {'ALERT' if v.flagged else 'ok'}")


## 4. Localise the change

The first deviating checkpoint bounds the search space. Every candidate gate in that prefix is mutated in a copy of the **reference** program, and the hypothesis that best reproduces the observation is ranked first. Evidence from up to three deviating checkpoints is fused.

In [ ]:
flagged = [heartbeat.by_id(v.cp_id) for v in verdict.verdicts if v.flagged]
localization = localize(
    program,                    # the REFERENCE program, never the tampered one
    flagged,
    baseline,
    backend,
    shots=baseline.shots,
    seed=202,
    true_gid=tamper.target_gid, # only used to label the result
    exact_sweep=True,
)

print(f"true gate : {tamper.target_gid}")
print(f"top-1     : {localization.top1_gid}  confidence {localization.confidence:.3f}"
      f"  ambiguous={localization.ambiguous}")
for candidate in localization.candidates[:3]:
    print(f"  #{candidate.rank} {candidate.gid}  {candidate.gate_description:<18}"
          f" D_TV={candidate.tvd_to_observed:.4f}  score={candidate.score:.3f}")


## 5. The evidence report

The same report is what the CLI, the SQLite history and the Streamlit dashboard consume: per-checkpoint distances, thresholds, p-values, the noise floor, the family-wise false-positive bound, the ranked candidates and any tie group.

In [ ]:
report = build_report(
    program,
    verdict,
    heartbeat=heartbeat,
    localization=localization,
    backend_description=backend.describe(),
)
print(report.to_text())
